# Credit Scorecard Development
## Notebook 2 — Weight of Evidence (WoE) & Information Value (IV) Analysis

WoE transforms raw features into a monotonic scale aligned with the log-odds of default,
making them directly suitable for logistic regression scorecards.

| IV Range | Predictive Power |
|----------|------------------|
| < 0.02   | Useless          |
| 0.02–0.1 | Weak             |
| 0.1–0.3  | Medium           |
| 0.3–0.5  | Strong           |
| > 0.5    | Suspicious       |

_Reference: Siddiqi, N. (2006). Credit Risk Scorecards._

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import generate_credit_data, train_test_split_time, WoEBinner

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

df = generate_credit_data(n_samples=15_000, random_state=42)
df_train, df_test = train_test_split_time(df, test_size=0.3)

features = ['age', 'annual_income', 'employment_years', 'loan_amount',
            'loan_term', 'credit_utilization', 'num_delinquencies',
            'num_credit_accounts', 'debt_to_income']

print(f'Train: {len(df_train):,} | Test: {len(df_test):,}')
print(f'Train default rate: {df_train["default"].mean():.2%}')

## 1. Fit WoE Binner on Training Data

In [ ]:
binner = WoEBinner(max_bins=10, min_bin_pct=0.05)
binner.fit(df_train[features], df_train['default'], features=features)

iv_summary = binner.iv_summary()
print('=== Information Value Summary ===')
print(iv_summary.to_string(index=False))

## 2. IV Bar Chart — Feature Predictive Power Ranking

In [ ]:
color_map = {'Useless':'#95a5a6','Weak':'#f39c12','Medium':'#2980b9',
             'Strong':'#27ae60','Suspicious':'#8e44ad'}
colors = iv_summary['interpretation'].map(color_map)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(iv_summary['feature'], iv_summary['iv'], color=colors, alpha=0.85)

for bar, iv_val in zip(bars, iv_summary['iv']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{iv_val:.4f}', va='center', fontsize=9)

for threshold, label, color in [(0.02,'Weak','#f39c12'),(0.1,'Medium','#2980b9'),
                                  (0.3,'Strong','#27ae60'),(0.5,'Suspicious','#8e44ad')]:
    ax.axvline(x=threshold, color=color, linestyle='--', alpha=0.6, linewidth=1)
    ax.text(threshold, ax.get_ylim()[0], label, color=color, fontsize=8, va='bottom')

ax.set_xlabel('Information Value (IV)')
ax.set_title('Feature Predictive Power — Information Value Ranking', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../plots/02_iv_ranking.png', bbox_inches='tight')
plt.show()

## 3. WoE Plots — Top Features

In [ ]:
top_features = iv_summary[iv_summary['iv'] >= 0.02]['feature'].tolist()
print(f'Features selected (IV >= 0.02): {top_features}')

for feat in top_features[:4]:  # show top 4
    fig = binner.plot_woe(feat, figsize=(12, 4))
    plt.savefig(f'../plots/02_woe_{feat}.png', bbox_inches='tight')
    plt.show()

## 4. WoE Transformation

In [ ]:
df_train_woe = binner.transform(df_train[features])
df_test_woe = binner.transform(df_test[features])

woe_features = [f'{f}_woe' for f in top_features]
print('WoE-transformed features sample:')
print(df_train_woe[woe_features].describe().round(4))

# WoE values should be monotonically related to default rate
print('\nCorrelation between WoE features and default:')
corr_woe = df_train_woe[woe_features].corrwith(df_train['default']).round(4)
print(corr_woe.sort_values(key=abs, ascending=False))